<div dir="rtl">

## 🤖 ایجنت نمترون روی کولب — نسخه‌ی وب


یه ایجنت کامل روی کولب رایگان (T4): چت + اجرای دستور با تایید تو + گشتن وب + مرورگر واقعی + فایل + **حافظه‌ی دائمی که از کارهاش یاد می‌گیره**. — رابطش یه صفحه‌ی وبه با لینک و پسورد.

**اجرا: از بالا به پایین، Run all**

- رانتایم باید **T4 GPU** باشه: Runtime → Change runtime type → T4.
- اجرای اول چند دقیقه ستاپ داره (دانلود مدل ۷٫۵ گیگ)؛ دفعات بعد سریع‌تره.
- نام کاربری و پسورد صفحه رو خودت ست می‌کنی: یا از قبل تو Colab Secrets با اسم‌های `NEMOTRON_WEB_USER` و `NEMOTRON_WEB_PASS` بذار، یا سلول اول ازت می‌پرسه. **اگه نام کاربری رو خالی بذاری، صفحه بدون لاگین باز می‌شه.**
</div>

<div dir="rtl">

## 📖  سلول تنظیم محیط

این سلول سه کار می‌کند: وصل‌کردن گوگل‌درایو، گرفتن نام کاربری و پسورد صفحه‌ی وب، و ساخت پوشه‌ی کاری همین مدل (`/content/nemotron-web`).

- **درایو = حافظه‌ی دائمی ایجنت.** مسیر `BOT_HOME` روی درایو ست می‌شود؛ ران‌بوک (دفترچه‌ی یادگیری) و تاریخچه‌ی گفتگو آنجا می‌مانند و وقتی رانتایم کولب عوض شود پاک نمی‌شوند.
- **لاگینِ قابل‌تنظیم:** اول از Secrets کولب خوانده می‌شود (کلیدهای `NEMOTRON_WEB_USER` و `NEMOTRON_WEB_PASS` — در هر سه نوت‌بوک همین نام‌ها)، و اگر نبود، از کاربر پرسیده می‌شود. اگر **هر دو خالی** باشند، صفحه بدون لاگین باز می‌شود؛ اگر هر دو پر باشند، لاگین فعال است. این انتخاب آگاهانه است: هر کس خودش تصمیم می‌گیرد صفحه‌اش قفل داشته باشد یا نه.

</div>


In [ ]:
import os

# --- حافظه‌ی دائمی روی گوگل درایو  ---
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["BOT_HOME"] = "/content/drive/MyDrive/nemotron-web"
    print("✅ Drive وصل شد — حافظه‌ی ایجنت دائمیه")
except Exception as e:
    print("⚠️ Drive وصل نشد؛ حافظه فقط همین سشن می‌مونه:", e)

# --- نام کاربری و پسورد صفحه‌ی وب (هر دو خالی = بدون لاگین) ---
user = ""
pw = ""
try:
    from google.colab import userdata
    user = userdata.get("NEMOTRON_WEB_USER") or ""
    pw = userdata.get("NEMOTRON_WEB_PASS") or ""
except Exception:
    pass
if not user:
    user = input("نام کاربری برای صفحه‌ی وب (خالی = بدون لاگین): ").strip()
if user and not pw:
    pw = input("پسورد صفحه‌ی وب (خالی = بدون لاگین): ").strip()
os.environ["WEB_USER"] = user
os.environ["WEB_PASS"] = pw
if user and pw:
    print("🔐 ورود فعاله — نام کاربری:", user)
else:
    print("🔓 ورود غیرفعاله — صفحه بدون نام کاربری و پسورد باز می‌شه")

os.makedirs("/content/nemotron-web", exist_ok=True)
os.chdir("/content/nemotron-web")
print("📁 پوشه‌ی کاری:", os.getcwd())
print("🔐 پسورد وب:", "تنظیم شد ✅" if pw else "نداره ⚠️")

Mounted at /content/drive
✅ Drive وصل شد — حافظه‌ی ایجنت دائمیه
نام کاربری برای صفحه‌ی وب (خالی = بدون لاگین): 
🔓 ورود غیرفعاله — صفحه بدون نام کاربری و پسورد باز می‌شه
📁 پوشه‌ی کاری: /content/nemotron-web
🔐 پسورد وب: نداره ⚠️


<div dir="rtl">

## 📖 setup.sh: نصب موازی

ایده‌ی اصلی این اسکریپت: **کارهای مستقل هم‌زمان اجرا می‌شوند** تا زمان ستاپ، جمعِ زمان تک‌تک کارها نباشد. هر کار لاگ جدا دارد (`.setup-logs/`) تا اگر چیزی خراب شد معلوم باشد کدام پله بوده.

- **باینری llama.cpp:** نسخه‌ی دقیق **b11878** از گیت‌هاب — «پین» کردن نسخه یعنی رفتار قابل‌تکرار روی هر ماشین. قبل از دانلود مدل، با `--version` سلامتش چک می‌شود.
- **دانلود مدل:** با curl و قابلیت ادامه (`-C -`) — اگر دانلود وسط راه قطع شود، اجرای دوباره از همان‌جا ادامه می‌دهد (فایل ۷.۵ گیگی).
- **پایتون:** نصب با **uv** (موازی و خیلی سریع‌تر از pip): گرادیو برای صفحه، httpx برای تماس با سرور مدل، curl_cffi برای فچ با اثر انگشت مرورگر واقعی، playwright برای ابزار مرورگر و بقیه.
- **مرورگر (Playwright):** عمداً دو قید دارد: (۱) **گیت‌شده روی pip** است، چون ماژول پلی‌رایت را لازم دارد و زودتر شروع کردنش خطا می‌دهد؛ (۲) **جدا (detached)** در پس‌زمینه نصب می‌شود، چون نصبش کند و پرنوسان است و نباید سرور مدل را معطل کند — ابزار مرورگر تا آماده‌شدنش پیام مناسب می‌دهد و چند دقیقه بعد خودکار آماده است.
- **تور:** با apt نصب می‌شود و فقط برای مسیر ناشناسِ ابزار fetch_url است؛ نبودنش فقط همان یک مسیر را خاموش می‌کند و بقیه‌ی ایجنت کامل کار می‌کند.
- **fix_cuda_libs —  ایمیج‌های جدید کولب سه کتابخانه‌ی CUDA (‏`libcuda.so.1`‎، ‏`libcublas.so.12` و کتابخانه‌ی PTX JIT درایور) را سر راه لودر نمی‌گذارند. نتیجه‌اش فاجعه‌ی ساکت است: باینری بدون هیچ خطایی روی **CPU** می‌افتد و ۲۰ برابر کند می‌شود. این تابع کتابخانه‌های خودِ ایمیج را پیدا و به پوشه‌ی llama لینک می‌کند و در پایان با `ldd` **اثبات** می‌کند که همه resolve شده‌اند (اگر cublas نسخه‌ی ۱۲ نبود، از ویل رسمی `nvidia-cublas-cu12` می‌آید).
- خروجی نهایی: فایل `config.env` (مسیر مدل) و خط `Setup OK` با زمان کل. کل اسکریپت **idempotent** است: اجرای دوباره روی رانتایم گرم چند ثانیه بیشتر طول نمی‌کشد.

</div>


In [ ]:
%%writefile setup.sh
#!/usr/bin/env bash
# ─────────────────────────────────────────────────────────────────────────────
# Colab setup for the Nemotron Nano 12B v2 agent bot — v4.
# Parallel where possible (binary + model + pip + tor together), but the
# browser step is GATED on pip: it needs the playwright module installed
# first (running it in parallel was the v2 bug — the browser job died with
# "No module named playwright" before pip had finished). Playwright gets
# the small HEADLESS SHELL first, with an automatic fallback to the full
# Chromium + system deps if the shell cannot launch on this image.
# v4 adds fix_cuda_libs(): current Colab images (python 3.13 / CUDA 13) no
# longer put libcuda / libcublas / the driver's PTX JIT library on the
# loader's path, which made the prebuilt bundle silently fall back to CPU
# (or abort at the first GPU compute). We link the image's own copies into
# llama/ and ASSERT with ldd that libggml-cuda fully resolves.
# Every step is idempotent: re-running on a warm runtime costs seconds.
# ─────────────────────────────────────────────────────────────────────────────
set -e
cd "$(dirname "$0")"
T0=$SECONDS

LLAMA_BUILD="b11878"
MODEL_URL="https://huggingface.co/bartowski/nvidia_NVIDIA-Nemotron-Nano-12B-v2-GGUF/resolve/main/nvidia_NVIDIA-Nemotron-Nano-12B-v2-Q4_K_M.gguf"
MODEL_FILE="$(basename "$MODEL_URL")"
mkdir -p llama models .setup-logs

job_binary() {
  if [ ! -x llama/llama-server ]; then
    curl -sfL -o llama/bin.tar.gz "https://github.com/ggml-org/llama.cpp/releases/download/$LLAMA_BUILD/llama-$LLAMA_BUILD-bin-ubuntu-cuda-12.8-x64.tar.gz"
    curl -sfL -o llama/cudart.tar.gz "https://github.com/ggml-org/llama.cpp/releases/download/$LLAMA_BUILD/cudart-llama-$LLAMA_BUILD-bin-ubuntu-cuda-12.8-x64.tar.gz"
    mkdir -p llama/extract
    tar -xzf llama/bin.tar.gz -C llama/extract
    tar -xzf llama/cudart.tar.gz -C llama/extract
    BIN_DIR="$(dirname "$(find llama/extract -name llama-server -type f | head -1)")"
    cp -r "$BIN_DIR"/. llama/
    find llama/extract \( -name 'libggml-cuda.so' -o -name 'libcudart.so*' \) -exec cp {} llama/ \;
    rm -rf llama/extract llama/bin.tar.gz llama/cudart.tar.gz
    chmod +x llama/llama-server
  fi
  LD_LIBRARY_PATH="$PWD/llama:${LD_LIBRARY_PATH:-}" ./llama/llama-server --version | head -1
}

job_model() {
  if [ ! -s "models/$MODEL_FILE" ]; then
    curl -sL -C - -o "models/$MODEL_FILE" "$MODEL_URL"
  fi
  ls -lh "models/$MODEL_FILE"
}

job_pip() {
  # uv installs the whole dependency set in parallel — much faster than pip.
  if ! command -v uv >/dev/null 2>&1; then
    python3 -m pip install -q uv || python3 -m pip install -q --break-system-packages uv
  fi
  if command -v uv >/dev/null 2>&1; then
    uv pip install --system -q gradio httpx curl_cffi beautifulsoup4 lxml pysocks playwright \
      || uv pip install --system --break-system-packages -q gradio httpx curl_cffi beautifulsoup4 lxml pysocks playwright
  else
    python3 -m pip install -q gradio httpx curl_cffi beautifulsoup4 lxml pysocks playwright \
      || python3 -m pip install -q --break-system-packages gradio httpx curl_cffi beautifulsoup4 lxml pysocks playwright
  fi
  echo "python deps OK"
}

browser_works() {
  python3 -c "
from playwright.sync_api import sync_playwright
with sync_playwright() as pw:
    b = pw.chromium.launch(args=['--no-sandbox']); b.close()
" >/dev/null 2>&1
}

job_browser() {
  # Wait until the playwright MODULE exists (the pip job provides it).
  for _ in $(seq 1 24); do
    python3 -c "import playwright" >/dev/null 2>&1 && break
    sleep 5
  done
  python3 -c "import playwright" >/dev/null 2>&1 || { echo "playwright module never appeared"; return 1; }
  if browser_works; then
    echo "chromium already works — skipped"
    return 0
  fi
  # Fast path: headless shell only (~110 MB, no apt).
  python3 -m playwright install --only-shell chromium
  if browser_works; then
    echo "chromium headless shell OK"
    return 0
  fi
  # Fallback: full Chromium + system libraries.
  echo "headless shell could not launch — installing full chromium with deps"
  python3 -m playwright install --with-deps chromium || python3 -m playwright install chromium
  browser_works
}

job_tor() {
  if command -v tor >/dev/null 2>&1; then
    echo "tor already installed — skipped"
  else
    apt-get update -qq && apt-get install -y -qq tor
  fi
}

# Detached browser-install mode (see below): same file, function-only run.
if [ "${1:-}" = "--browser-only" ]; then
  mkdir -p .setup-logs
  job_browser
  exit $?
fi

echo "==> parallel setup: binary + model (7.5 GB) + python deps + tor"
job_binary > .setup-logs/binary.log 2>&1 & P1=$!
job_model  > .setup-logs/model.log  2>&1 & P2=$!
job_pip    > .setup-logs/pip.log    2>&1 & P3=$!
job_tor    > .setup-logs/tor.log    2>&1 & P5=$!

FAIL=0
# pip gates the browser step (the browser needs the playwright module).
if ! wait "$P3"; then echo "❌ step FAILED: pip — log:"; tail -15 .setup-logs/pip.log; FAIL=1; fi
if [ "$FAIL" = "0" ]; then
  # The browser is ONE tool of eleven, and its install is the slowest,
  # flakiest step (Playwright CDN + a silent apt phase that looks like a
  # hang). NEVER let it delay the model server: if Chromium already
  # works we skip in a second; otherwise the install runs DETACHED and
  # setup moves on immediately. browser_open reports per-call until
  # the browser is ready (usually a few minutes later).
  if browser_works; then
    echo "==> browser step: chromium already works — skipped"
  else
    setsid nohup bash setup.sh --browser-only >> .setup-logs/browser.log 2>&1 &
    echo "==> browser install running in BACKGROUND — not blocking the server (log: .setup-logs/browser.log)"
  fi
fi
for spec in "binary $P1" "model $P2"; do
  set -- $spec
  if ! wait "$2"; then echo "❌ step FAILED: $1 — log:"; tail -15 ".setup-logs/$1.log"; FAIL=1; fi
done
wait "$P5" 2>/dev/null || echo "⚠️ tor unavailable (bot still works; only fetch_url's Tor route is off)"
[ "$FAIL" = "1" ] && exit 1

fix_cuda_libs() {
  # Link the libraries the prebuilt CUDA bundle needs but this image does not
  # put on the default search path. llama/ is on LD_LIBRARY_PATH for every
  # launch, so links there fix the server, server_cmd.sh and the watchdog.
  local name src roots="/usr/lib64-nvidia /usr/lib64 /usr/local /usr/lib"
  for name in libcuda.so.1 libnvidia-ptxjitcompiler.so.1 libcublas.so.12 libcublasLt.so.12; do
    if [ ! -e "llama/$name" ]; then
      # exact SONAME first, then the unversioned name (some images ship
      # libcuda.so without the .so.1 symlink — same library, link it under
      # the name the binary asks for). Never the CUDA stubs dir: that
      # libcuda is a link-time dummy that fails at runtime.
      src="$(find $roots -name "$name" -not -path '*stubs*' 2>/dev/null | head -1)"
      if [ -z "$src" ]; then
        src="$(find $roots -name "${name%.*}" -not -path '*stubs*' 2>/dev/null | head -1)"
      fi
      if [ -n "$src" ]; then ln -s "$src" "llama/$name"; echo "  linked $name -> $src"; fi
    fi
  done
  # Newer Colab images ship only CUDA 13's libcublas (.so.13) via torch; the
  # prebuilt bundle needs the CUDA 12 SONAMEs. Pull the versioned cu12 wheel
  # and link from it (this bit a fresh runtime on 2026-10-27 evening).
  if [ ! -e llama/libcublas.so.12 ]; then
    echo "  libcublas.so.12 missing — installing nvidia-cublas-cu12"
    python3 -m pip install -q nvidia-cublas-cu12 2>/dev/null || pip install -q nvidia-cublas-cu12 || true
    for name in libcublas.so.12 libcublasLt.so.12; do
      if [ ! -e "llama/$name" ]; then
        src="$(find /usr/local/lib /usr/lib -name "$name" 2>/dev/null | head -1)"
        if [ -n "$src" ]; then ln -s "$src" "llama/$name"; echo "  linked $name -> $src"; fi
      fi
    done
  fi
  local missing
  missing="$(LD_LIBRARY_PATH="$PWD/llama" ldd llama/libggml-cuda.so 2>/dev/null | grep 'not found' || true)"
  if [ -n "$missing" ]; then
    echo "❌ CUDA libraries still unresolved:"; echo "$missing"; return 1
  fi
  echo "  CUDA library check OK (ldd clean)"
}

echo "==> CUDA library fix-ups"
if ! fix_cuda_libs; then exit 1; fi

for spec in binary model pip browser; do echo "    [$spec] $(tail -1 ".setup-logs/$spec.log")"; done
echo "MODEL_PATH=$PWD/models/$MODEL_FILE" > config.env
echo "✅ Setup OK in $((SECONDS - T0))s"

Writing setup.sh


<div dir="rtl">

## 📖  start_server.sh: روشن‌کردن سرور مدل

- اول `setup.sh` را دوباره صدا می‌زند — چون idempotent است، روی رانتایم تازه کارش را کامل می‌کند و روی رانتایم گرم در چند ثانیه رد می‌شود. پس این سلول به‌تنهایی هم برای بالا آوردن کافی است.
- **ری‌استارت تمیز:** هر سرور قبلی روی پورت ۸۰۹۰ کشته می‌شود. دو سرور هم‌زمان یعنی جنگ بر سر پورت و حافظه‌ی GPU و مرگ هر دو — این خط جلوی همان را می‌گیرد.
- **`server_cmd.sh`:** خط دقیق لانچ سرور در یک فایل جدا ذخیره می‌شود تا واچ‌داگِ صفحه‌ی وب هم برای احیای سرور دقیقاً از همان فایل استفاده کند (یک منبع حقیقت واحد).
- فلگ‌های لانچ: `--jinja` قالب چت و فراخوانی بومی ابزارِ مدل را فعال می‌کند؛ `-ngl 99` یعنی آفلود کامل همه‌ی لایه‌ها روی GPU؛ `-c 32768` پنجره‌ی کانتکست این مدل؛ `-b 256 -ub 256` اندازه‌ی بچ محافظه‌کار برای جا شدن مطمئن روی T4؛ پورت **۸۰۹۰** چون ۸۰۸۰ را خود کولب اشغال کرده.
- **حلقه‌ی انتظار با چک مرگ:** هر ۳ ثانیه اندپوینت هلث چک می‌شود و هم‌زمان زنده‌بودن پروسه هم بررسی می‌شود؛ اگر سرور موقع لود بمیرد، سلول دمِ لاگ را چاپ می‌کند و متوقف می‌شود — خطا هیچ‌وقت بی‌صدا نمی‌ماند.
- **وارم‌آپ:** اولین درخواست واقعی روی CUDA یک کامپایل یک‌باره‌ی PTX JIT دارد (~۳۰ تا ۶۰ ثانیه). اینجا با یک درخواست ساختگی همان اول پرداختش می‌کنیم تا اولین جواب کاربر سریع باشد.

</div>


In [ ]:
%%writefile start_server.sh
#!/usr/bin/env bash
# ─────────────────────────────────────────────────────────────────────────────
# Launcher for the WEB agent (called by the Colab notebook):
#   1. setup.sh (idempotent — fast on a warm runtime)
#   2. (re)start llama-server on port 8090, health-wait WITH death check
# The web app itself is started by its own notebook cell afterwards.
# Logs: server.log in this folder.
# ─────────────────────────────────────────────────────────────────────────────
set -e
cd "$(dirname "$0")"

export BOT_HOME="${BOT_HOME:-$PWD/data}"

bash setup.sh
source config.env   # MODEL_PATH=...

# Clean restart: stop whatever a previous run left behind.
[ -f server.pid ] && kill "$(cat server.pid)" 2>/dev/null || true
pkill -f "llama-server.*--port 8090" 2>/dev/null || true
sleep 1

# The exact server launch line lives in server_cmd.sh — the bot's watchdog
# re-uses this same file to resurrect the server if it ever dies.
cat > server_cmd.sh <<EOF
#!/usr/bin/env bash
cd "$PWD"
export LD_LIBRARY_PATH="$PWD/llama:/usr/lib64-nvidia:\${LD_LIBRARY_PATH:-}"
exec ./llama/llama-server -m "$MODEL_PATH" --jinja -ngl 99 \\
  -c 32768 -b 256 -ub 256 \\
  --host 127.0.0.1 --port 8090 --alias nemotron-nano-12b >> server.log 2>&1
EOF
chmod +x server_cmd.sh

echo "==> starting llama-server (Nemotron Nano 12B v2, Q4_K_M, T4)..."
setsid nohup bash server_cmd.sh >/dev/null 2>&1 &
echo $! > server.pid

echo "==> waiting for the model to load (death-checked)..."
for i in $(seq 1 120); do
  if curl -s http://127.0.0.1:8090/health >/dev/null 2>&1; then
    echo "    server READY after ~$((i * 3))s"
    break
  fi
  if ! pgrep -f "llama-server.*--port 8090" >/dev/null; then
    echo "    SERVER DIED — last log lines:"; tail -20 server.log; exit 1
  fi
  sleep 3
  if [ "$i" = "120" ]; then echo "    TIMEOUT — last log lines:"; tail -20 server.log; exit 1; fi
done

echo "==> warming up (one-time CUDA PTX JIT compile, ~30-60s)..."
curl -s -m 300 http://127.0.0.1:8090/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"model":"nemotron-nano-12b","messages":[{"role":"user","content":"OK"}],"max_tokens":1}' \
  >/dev/null 2>&1 || true
echo "    warm — the first web answer will be fast"

echo "✅ MODEL SERVER IS RUNNING — http://127.0.0.1:8090"
echo "   server log: tail -f $PWD/server.log"
echo "   next step: run the web-app cell below."

Writing start_server.sh


<div dir="rtl">

## 📖 web_app.py: مغز ایجنت و صفحه‌ی وب

بزرگ‌ترین فایل نوت‌بوک است و دو لایه دارد: **هسته‌ی ایجنت** (منطق، مستقل از ظاهر) و **لایه‌ی وب** (صفحه‌ی گرادیو). تور بخش‌هایش:

- **پل وب:** هسته‌ی ایجنت در یک **ترد جدا** با حلقه‌ی asyncio خودش اجرا می‌شود و گرادیو از طریق یک دیکشنری مشترک (`STATE`) با آن حرف می‌زند. دلیلش: اجرای یک تسک ایجنت ممکن است دقیقه‌ها طول بکشد و رابط کاربری نباید قفل شود؛ وضعیت (متن در حال تولید، انتظار تایید، فایل آماده) از همین STATE خوانده می‌شود.
- **ابزارها (۱۱ تا):** اجرای دستور شل، فچ صفحه و درخواست HTTP (با امکان عبور از تور)، مرورگر واقعی با اسکرین‌شات، خواندن/نوشتن/لیست فایل (**با گارد مسیر** — بیرون از پوشه‌های مجاز رد می‌شود)، ارسال فایل، ساعت، و حافظه‌ی به‌خاطر سپردن/بازیابی.
- **گیت تایید:** فقط اجرای دستور شل نیازمند تایید است؛ نوار ✅/❌ زیر چت ظاهر می‌شود و تمام‌شدن مهلتش یعنی «رد». بقیه‌ی ابزارها گیت ندارند — تصمیم طراحی: اصطکاک فقط همان‌جا که خطر واقعی هست. (در پنل تنظیمات، حالت تایید را می‌شود کلاً خاموش کرد.)
- **حلقه‌ی ایجنت و ناج اجرا:** حلقه سقف گام دارد تا گیر نکند. اگر مدل برای یک کار اجرایی (بفرست، آپلود کن، اجرا کن) به‌جای صدا زدن ابزار فقط حرف بزند، حلقه یک بار آن جواب را رد می‌کند و ساختاری مجبورش می‌کند ابزار را صدا بزند. درس مهم: رفتار مدل را با **هارنس** درست کنید، نه با خواهش در پرامپت.
- **ران‌بوک — یادگیری از تجربه:** بعد از هر تسک چندابزاری، یک فراخوانی جدا خلاصه می‌کند «چه روشی جواب داد و چه روشی نه» و در فایل JSONL روی درایو ذخیره می‌شود؛ این خلاصه‌ها در پرامپت سیستمِ همه‌ی پیام‌های بعدی تزریق می‌شوند. ورودی‌ای که بعداً نقض شود استرایک می‌خورد و با دو استرایک حذف می‌شود؛ اطلاعات محرمانه قبل از ذخیره پاک می‌شوند. (مثال زنده‌اش در تست ۱۷ باتری دیده می‌شود.)
- **تفکر این مدل:** با تگ‌های `/think` و `/no_think` در پرامپت سیستم کنترل می‌شود و از پنل تنظیمات صفحه هم قابل خاموش/روشن‌کردن است.
- **تنظیمات زنده:** دما، top-p و top-k، سقف توکن خروجی (پیش‌فرض این مدل: 1024)، کلید تفکر و حتی متن پرامپت سیستم از پنل پایین صفحه **بدون ری‌استارت** عوض می‌شوند.
- **واچ‌داگ:** هر ۳۰ ثانیه هلث سرور مدل را چک می‌کند؛ اما قبل از احیای سرور، اول مطمئن می‌شود پروسه‌ای زنده نیست — مدل‌های بزرگ چند دقیقه «مرده» به نظر می‌رسند و احیای اشتباه یعنی جنگ دو سرور روی یک GPU.
- **رابط کاربری:** چت‌محور است؛ اسکرین‌شات و فایل **داخل خود چت** تحویل داده می‌شوند (کامپوننت داخل آکاردئونِ بسته روی موبایل خراب رندر می‌شد). ارتفاع چت روی موبایل ثابت است و روی دسکتاپ با قد صفحه رشد می‌کند تا باکس پیام پایین صفحه بماند؛ جهت صفحه راست‌به‌چپ است و کدها جزیره‌ی چپ‌به‌راست خودشان را دارند.

</div>


In [ ]:
%%writefile web_app.py
#!/usr/bin/env python3
"""
Nemotron Nano 12B v2 — WEB agent for Google Colab (free T4).

Same agent core as the Telegram bot (same tools, same confirm gate, same
Drive runbook, same history) with the Telegram face replaced by a Gradio
web UI: streaming chat, live generation parameters (temperature / top-p /
top-k / max tokens / thinking / extra system instruction), an on-page
approve-once / approve-all / deny bar for shell commands, inline browser
screenshots, file downloads, and memory / status / reset / wipe buttons.

Config via environment variables:
  BOT_HOME    data folder for history.json + runbook.jsonl (Drive = permanent)
  WORK_DIR    scratch folder for the file tools + screenshots
  SERVER_URL  llama-server base (default http://127.0.0.1:8090/v1)
  WEB_USER / WEB_PASS   login for the web page (set WEB_PASS from Colab
              Secrets — the page can run shell commands after approval, so
              an open link is NOT safe)
  WEB_PORT    default 7860
  THINKING    default for the thinking checkbox ("1" on, "0" off)
"""

import asyncio
import json
import os
import re
import subprocess
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path

import httpx

try:
    BASE_DIR = Path(os.path.dirname(os.path.abspath(__file__)))
except NameError:  # running inside a notebook cell
    BASE_DIR = Path.cwd()
BOT_TOKEN = os.environ.get("BOT_TOKEN", "")
SERVER_URL = os.environ.get("SERVER_URL", "http://127.0.0.1:8090/v1").rstrip("/")
MODEL = os.environ.get("MODEL_ALIAS", "nemotron-nano-12b")
DATA_HOME = Path(os.environ.get("BOT_HOME", str(BASE_DIR / "data")))
WORK_DIR = Path(os.environ.get("WORK_DIR", str(BASE_DIR / "work")))
DATA_HOME.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

THINKING = os.environ.get("THINKING", "1") != "0"

TEMPERATURE = 0.6   # NVIDIA official sampling for Nemotron Nano 12B v2
TOP_P = 0.95
MAX_TOKENS = 4096
MAX_STEPS = 10              # harness step budget per task (not an anti-loop:
                            # the model self-corrects freely inside it)
CONFIRM_TIMEOUT = 180       # seconds to wait for a run_command approval
HISTORY_TURNS = 40
MAX_HISTORY_CHARS = 32000
HISTORY_FILE = DATA_HOME / "history.json"
RUNBOOK_FILE = DATA_HOME / "runbook.jsonl"
RUNBOOK_INJECT_CHARS = 4500
TOOL_OUT_CAP = 8000

TEHRAN = timezone(timedelta(hours=3, minutes=30))
THINKING_TEXT = "🧠 دارم فکر می‌کنم..."
WORKING_TEXT = "⏳ دارم روش کار می‌کنم..."

BASE_SYSTEM = (
    "تو یه دستیار ایجنت هستی که روی یه سرور لینوکسی (Google Colab) با دسترسی به ابزارها کار می‌کنی. "
    "برای کارهای چندمرحله‌ای از ابزارهات استفاده کن و نتیجه رو بر اساس خروجی واقعی ابزارها گزارش بده، نه حدس. "
    "اگه یه مسیر fail شد، مسیر دیگه‌ای رو امتحان کن. "
    "همیشه به همون زبونی جواب بده که کاربر نوشته. "
    "جواب نهایی باید خلاصه و دقیق باشه و عددها و نتایج واقعی ابزارها توش باشه."
)

# --------------------------------------------------------------------------
# Chat history (Daytona pattern: persisted, pair-trimmed, char-budgeted)
# --------------------------------------------------------------------------
histories: dict[int, list[dict]] = {}


def _load_histories() -> None:
    try:
        with open(HISTORY_FILE, encoding="utf-8") as f:
            for chat_id, msgs in json.load(f).items():
                histories[int(chat_id)] = msgs
    except Exception:
        pass


def _save_histories() -> None:
    try:
        with open(HISTORY_FILE, "w", encoding="utf-8") as f:
            json.dump(histories, f, ensure_ascii=False)
    except Exception:
        pass


_load_histories()

# --------------------------------------------------------------------------
# Runbook — the permanent, structured experience memory
# --------------------------------------------------------------------------


def load_runbook() -> list[dict]:
    entries = []
    try:
        with open(RUNBOOK_FILE, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    try:
                        entries.append(json.loads(line))
                    except ValueError:
                        pass
    except Exception:
        pass
    return entries


def save_runbook(entries: list[dict]) -> None:
    try:
        tmp = RUNBOOK_FILE.with_suffix(".tmp")
        with open(tmp, "w", encoding="utf-8") as f:
            for e in entries:
                f.write(json.dumps(e, ensure_ascii=False) + "\n")
        tmp.replace(RUNBOOK_FILE)
    except Exception:
        pass


def render_runbook(entries: list[dict], cap: int = RUNBOOK_INJECT_CHARS) -> str:
    """Compact text block injected into the system prompt, newest first."""
    lines = []
    total = 0
    for e in reversed(entries):
        parts = [f"تسک: {e.get('task', '')}"]
        if e.get("worked"):
            parts.append("جواب داد: " + "؛ ".join(e["worked"]))
        if e.get("failed"):
            parts.append("جواب نداد: " + "؛ ".join(e["failed"]))
        line = "• " + " | ".join(parts)
        if total + len(line) > cap:
            break
        lines.append(line)
        total += len(line)
    return "\n".join(lines)


_SECRET_PATTERNS = [
    re.compile(r"\d{6,}:[A-Za-z0-9_-]{30,}"),          # Telegram bot tokens
    re.compile(r"(?i)(token|api[_-]?key|password|secret)\s*[=:]\s*\S+"),
    re.compile(r"\b[A-Fa-f0-9]{32,}\b"),                 # long hex strings
]


def scrub(text: str) -> str:
    for pat in _SECRET_PATTERNS:
        text = pat.sub("[scrubbed]", text)
    return text


def _norm_task(task: str) -> str:
    return re.sub(r"\s+", " ", task.strip().lower())[:80]

# --------------------------------------------------------------------------
# Tools — schemas (sent to the model) + implementations (run by the harness)
# --------------------------------------------------------------------------

TOOLS = [
    {"type": "function", "function": {
        "name": "run_command",
        "description": "Execute a bash command on this Linux machine (Google Colab) and return stdout+stderr and the exit code. Use it for system info, installing packages, running scripts, calculations, downloads — anything a shell can do. The user approves each command right here in the web page (when the approval setting is on).",
        "parameters": {"type": "object", "properties": {
            "command": {"type": "string", "description": "The bash command to run"},
            "timeout": {"type": "integer", "description": "Seconds before the command is killed (default 90, max 600)"}},
            "required": ["command"]}}},
    {"type": "function", "function": {
        "name": "fetch_url",
        "description": "Fetch a web page or API URL with a real browser fingerprint (defeats most bot checks) and return its readable text. Set use_tor=true only if the site is geo-blocked or unreachable directly.",
        "parameters": {"type": "object", "properties": {
            "url": {"type": "string"},
            "use_tor": {"type": "boolean", "description": "Route through the local Tor proxy (slower)"}},
            "required": ["url"]}}},
    {"type": "function", "function": {
        "name": "http_request",
        "description": "Make a generic HTTP request (any method, custom headers, optional JSON body) and return the status code and response text. Use for APIs that fetch_url cannot handle.",
        "parameters": {"type": "object", "properties": {
            "method": {"type": "string", "description": "GET, POST, PUT, DELETE, ..."},
            "url": {"type": "string"},
            "headers": {"type": "object", "description": "Optional header map"},
            "json_body": {"type": "string", "description": "Optional JSON string sent as the request body"}},
            "required": ["method", "url"]}}},
    {"type": "function", "function": {
        "name": "browser_open",
        "description": "Open a URL in a real headless Chromium browser (executes JavaScript), return the page's visible text, and attach a screenshot of the page to this conversation. Use only when fetch_url fails or the page needs JavaScript.",
        "parameters": {"type": "object", "properties": {
            "url": {"type": "string"}}, "required": ["url"]}}},
    {"type": "function", "function": {
        "name": "read_file",
        "description": "Read a text file and return its content (truncated). Paths are relative to the work folder unless absolute.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]}}},
    {"type": "function", "function": {
        "name": "write_file",
        "description": "Write text content to a file (creates or overwrites). Paths are relative to the work folder unless absolute.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"},
            "content": {"type": "string"}}, "required": ["path", "content"]}}},
    {"type": "function", "function": {
        "name": "list_files",
        "description": "List files and folders at a path (default: the work folder).",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "Folder to list (default '.')"}}, "required": []}}},
    {"type": "function", "function": {
        "name": "send_file",
        "description": "Send a file from this machine to the user in this web page: it appears in the conversation as a download. When the user asks for a file, create it and then ALWAYS call this tool to deliver it — never ask permission to send it, and never claim a file was sent unless you called this tool.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]}}},
    {"type": "function", "function": {
        "name": "get_datetime",
        "description": "Return the current date and time (UTC and Tehran). Models do not know 'now' — call this whenever the task depends on the current time.",
        "parameters": {"type": "object", "properties": {}, "required": []}}},
    {"type": "function", "function": {
        "name": "remember",
        "description": "Save an important fact or lesson into the permanent experience memory (runbook) so future sessions know it. The harness also records task experiences automatically; use this for explicit facts the user tells you or lessons worth keeping verbatim.",
        "parameters": {"type": "object", "properties": {
            "text": {"type": "string"}}, "required": ["text"]}}},
    {"type": "function", "function": {
        "name": "recall",
        "description": "Search the permanent experience memory (runbook) for entries matching a query and return them.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]}}},
]


class TaskCtx:
    """Everything a tool needs while one user task runs: the UI callbacks
    (confirm gate, photo/file delivery), the cancel flag, the step journal
    that later becomes a runbook entry."""

    def __init__(self, ui, chat_id: int):
        self.ui = ui
        self.chat_id = chat_id
        self.cancel = asyncio.Event()
        self.approve_all = False
        self.journal: list[dict] = []
        self.proc = None  # running subprocess, killed on cancel

    def cancelled(self) -> bool:
        return self.cancel.is_set()


def _guard_path(p: str) -> Path | None:
    path = Path(p).expanduser()
    if not path.is_absolute():
        path = WORK_DIR / path
    try:
        resolved = path.resolve()
    except Exception:
        return None
    roots = [WORK_DIR.resolve(), DATA_HOME.resolve(), BASE_DIR.resolve(),
             Path("/content").resolve()]
    if any(str(resolved) == str(r) or str(resolved).startswith(str(r) + "/") for r in roots):
        return resolved
    return None


async def tool_run_command(ctx: TaskCtx, command: str, timeout: int = 90) -> str:
    if getattr(ctx, "confirm_mode", True):
        decision = await ctx.ui.confirm(command)
        if decision != "once":
            return "ERROR: user denied this command. Do not retry it; find another way or ask the user."
    timeout = max(5, min(int(timeout or 90), 600))
    try:
        proc = await asyncio.create_subprocess_shell(
            command, cwd=str(WORK_DIR),
            stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.STDOUT)
        ctx.proc = proc
        try:
            out, _ = await asyncio.wait_for(proc.communicate(), timeout=timeout)
        except asyncio.TimeoutError:
            proc.kill()
            return f"ERROR: command timed out after {timeout}s and was killed."
        if ctx.cancelled():
            try:
                proc.kill()
            except Exception:
                pass
            return "ERROR: task cancelled by the user."
        text = out.decode("utf-8", "replace")
        return f"exit={proc.returncode}\n{text[-6000:]}"
    except Exception as e:
        return f"ERROR: {e}"
    finally:
        ctx.proc = None


_tor_started = False


def _fetch_url_sync(url: str, use_tor: bool) -> str:
    global _tor_started
    proxies = None
    if use_tor:
        if not _tor_started:
            try:
                subprocess.Popen(["tor"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                _tor_started = True
                time.sleep(6)  # let Tor build its first circuits
            except Exception as e:
                return f"ERROR: could not start Tor: {e}"
        proxies = {"http": "socks5h://127.0.0.1:9050", "https": "socks5h://127.0.0.1:9050"}
    try:
        from curl_cffi import requests as creq
        r = creq.get(url, impersonate="chrome", timeout=40, proxies=proxies)
        status, raw, ctype = r.status_code, r.text, r.headers.get("content-type", "")
    except Exception:
        import httpx as _hx
        kw = {"proxy": proxies["https"]} if proxies else {}
        with _hx.Client(timeout=40, follow_redirects=True, **kw) as c:
            r = c.get(url, headers={"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/124 Safari/537.36"})
        status, raw, ctype = r.status_code, r.text, r.headers.get("content-type", "")
    if "html" in ctype or raw.lstrip().lower().startswith(("<!doctype", "<html")):
        try:
            from bs4 import BeautifulSoup
            text = BeautifulSoup(raw, "lxml").get_text("\n", strip=True)
        except Exception:
            text = re.sub(r"<[^>]+>", " ", raw)
    else:
        text = raw
    return f"HTTP {status}\n{text[:TOOL_OUT_CAP]}"


async def tool_fetch_url(ctx: TaskCtx, url: str, use_tor: bool = False) -> str:
    try:
        return await asyncio.get_running_loop().run_in_executor(
            None, _fetch_url_sync, url, bool(use_tor))
    except Exception as e:
        return f"ERROR: {e}"


async def tool_http_request(ctx: TaskCtx, method: str, url: str,
                            headers: dict | None = None,
                            json_body: str | None = None) -> str:
    try:
        data = None
        if json_body:
            try:
                data = json.loads(json_body)
            except ValueError:
                data = json_body
        async with httpx.AsyncClient(timeout=40, follow_redirects=True) as c:
            r = await c.request(method.upper(), url, headers=headers or {},
                                json=data if isinstance(data, (dict, list)) else None,
                                content=data if isinstance(data, str) else None)
        return f"HTTP {r.status_code}\n{r.text[:TOOL_OUT_CAP]}"
    except Exception as e:
        return f"ERROR: {e}"


async def tool_browser_open(ctx: TaskCtx, url: str) -> str:
    try:
        from playwright.async_api import async_playwright
    except ImportError:
        return "ERROR: playwright is not installed. Install it with run_command: python3 -m playwright install --with-deps chromium"
    shot = WORK_DIR / f"shot_{int(time.time())}.png"
    try:
        async with async_playwright() as pw:
            browser = await pw.chromium.launch(args=["--no-sandbox"])
            page = await browser.new_page(viewport={"width": 1280, "height": 900})
            await page.goto(url, wait_until="domcontentloaded", timeout=45000)
            await page.wait_for_timeout(2500)
            text = await page.evaluate("document.body ? document.body.innerText : ''")
            await page.screenshot(path=str(shot))
            await browser.close()
    except Exception as e:
        return f"ERROR: browser failed: {e}"
    await ctx.ui.photo(shot, "📸 اسکرین‌شات صفحه‌ای که باز کردم")
    return f"Page text from {url}:\n{(text or '')[:TOOL_OUT_CAP]}\n[a screenshot of the page was sent to the user]"


async def tool_read_file(ctx: TaskCtx, path: str) -> str:
    p = _guard_path(path)
    if p is None:
        return "ERROR: path is outside the allowed folders."
    try:
        return p.read_text(encoding="utf-8", errors="replace")[:TOOL_OUT_CAP]
    except Exception as e:
        return f"ERROR: {e}"


async def tool_write_file(ctx: TaskCtx, path: str, content: str) -> str:
    p = _guard_path(path)
    if p is None:
        return "ERROR: path is outside the allowed folders."
    try:
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(content, encoding="utf-8")
        return f"OK: wrote {len(content)} chars to {p}"
    except Exception as e:
        return f"ERROR: {e}"


async def tool_list_files(ctx: TaskCtx, path: str = ".") -> str:
    p = _guard_path(path)
    if p is None:
        return "ERROR: path is outside the allowed folders."
    try:
        items = []
        for child in sorted(p.iterdir()):
            kind = "dir " if child.is_dir() else "file"
            size = child.stat().st_size if child.is_file() else 0
            items.append(f"{kind} {child.name} ({size} B)")
        return "\n".join(items) or "(empty)"
    except Exception as e:
        return f"ERROR: {e}"


async def tool_send_file(ctx: TaskCtx, path: str) -> str:
    p = _guard_path(path)
    if p is None or not p.is_file():
        return "ERROR: file not found or outside the allowed folders."
    if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp", ".gif"}:
        await ctx.ui.photo(p, f"🖼 {p.name}")
    else:
        await ctx.ui.file(p)
    return f"OK: sent {p.name} to the user."


async def tool_get_datetime(ctx: TaskCtx) -> str:
    now = datetime.now(timezone.utc)
    return (f"UTC: {now:%Y-%m-%d %H:%M:%S} | Tehran: {now.astimezone(TEHRAN):%Y-%m-%d %H:%M:%S} "
            f"({now.astimezone(TEHRAN):%A})")


async def tool_remember(ctx: TaskCtx, text: str) -> str:
    entries = load_runbook()
    entries.append({
        "ts": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "task": "یادداشت صریح (explicit note)",
        "worked": [scrub(text)[:500]], "failed": [], "strikes": 0,
    })
    save_runbook(entries)
    return "OK: saved to the permanent runbook."


async def tool_recall(ctx: TaskCtx, query: str) -> str:
    q = query.lower()
    hits = [e for e in load_runbook()
            if q in json.dumps(e, ensure_ascii=False).lower()]
    if not hits:
        return "No matching entries in the runbook."
    return render_runbook(hits, cap=4000)


TOOL_FUNCS = {
    "run_command": tool_run_command,
    "fetch_url": tool_fetch_url,
    "http_request": tool_http_request,
    "browser_open": tool_browser_open,
    "read_file": tool_read_file,
    "write_file": tool_write_file,
    "list_files": tool_list_files,
    "send_file": tool_send_file,
    "get_datetime": tool_get_datetime,
    "remember": tool_remember,
    "recall": tool_recall,
}

# --------------------------------------------------------------------------
# Model I/O — streaming chat completion with native tool calls
# --------------------------------------------------------------------------

_THINK_BLOCK = re.compile(r"<think>.*?</think>", re.DOTALL)
_TOOL_CALL_BLOCK = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.DOTALL)


def visible_text(content: str) -> str:
    """Final-answer text only: think blocks removed (an unclosed one cuts the
    rest), raw text tool-call blocks removed."""
    text = _THINK_BLOCK.sub("", content)
    open_idx = text.rfind("<think>")
    if open_idx != -1:
        text = text[:open_idx]
    text = _TOOL_CALL_BLOCK.sub("", text)
    return text.strip()


def parse_text_tool_calls(content: str) -> list[dict]:
    """Fallback: some templates emit tool calls as <tool_call>{json}</tool_call>
    text instead of structured calls. Parse them into the same shape."""
    calls = []
    for i, m in enumerate(_TOOL_CALL_BLOCK.finditer(content)):
        try:
            obj = json.loads(m.group(1))
        except ValueError:
            continue
        name = obj.get("name")
        if not name:
            continue
        args = obj.get("arguments", {})
        calls.append({
            "id": f"text-{int(time.time())}-{i}",
            "name": name,
            "args": args if isinstance(args, dict) else {},
        })
    return calls


async def stream_completion(messages, tools, on_progress, max_tokens=MAX_TOKENS, params=None):
    """One streaming completion. Returns (content, reasoning, tool_calls,
    finish_reason) where tool_calls is a list of {id, name, args(dict)}."""
    payload = {
        "model": MODEL,
        "messages": messages,
        "tools": tools,
        "tool_choice": "auto",
        "temperature": (params or {}).get("temperature", TEMPERATURE),
        "top_p": (params or {}).get("top_p", TOP_P),
        "max_tokens": (params or {}).get("max_tokens", max_tokens),
        "stream": True,
    }
    if params and params.get("top_k"):
        payload["top_k"] = int(params["top_k"])
    if params and params.get("repeat_penalty"):
        payload["repeat_penalty"] = float(params["repeat_penalty"])
    content, reasoning = "", ""
    tc_by_index: dict[int, dict] = {}
    finish = ""
    async with httpx.AsyncClient(timeout=3600, trust_env=False) as client:
        async with client.stream("POST", f"{SERVER_URL}/chat/completions", json=payload) as resp:
            resp.raise_for_status()
            async for line in resp.aiter_lines():
                if not line.startswith("data:"):
                    continue
                data = line[5:].strip()
                if data == "[DONE]":
                    break
                try:
                    choice = json.loads(data)["choices"][0]
                except (ValueError, KeyError, IndexError):
                    continue
                delta = choice.get("delta") or {}
                if choice.get("finish_reason"):
                    finish = choice["finish_reason"]
                if delta.get("reasoning_content"):
                    reasoning += delta["reasoning_content"]
                if delta.get("content"):
                    content += delta["content"]
                for tc in delta.get("tool_calls") or []:
                    idx = tc.get("index", 0)
                    slot = tc_by_index.setdefault(idx, {"id": "", "name": "", "args_str": ""})
                    if tc.get("id"):
                        slot["id"] = tc["id"]
                    fn = tc.get("function") or {}
                    if fn.get("name"):
                        slot["name"] = fn["name"]
                    if fn.get("arguments"):
                        slot["args_str"] += fn["arguments"]
                if on_progress:
                    await on_progress(content, reasoning)
    tool_calls = []
    for idx in sorted(tc_by_index):
        slot = tc_by_index[idx]
        try:
            args = json.loads(slot["args_str"]) if slot["args_str"] else {}
        except ValueError:
            args = {}
        if slot["name"]:
            tool_calls.append({"id": slot["id"] or f"call-{idx}",
                               "name": slot["name"], "args": args})
    if not tool_calls:
        tool_calls = parse_text_tool_calls(content)
    return content, reasoning, tool_calls, finish


async def plain_completion(messages, max_tokens=600):
    """Non-streaming completion (used for runbook summaries)."""
    payload = {
        "model": MODEL, "messages": messages,
        "temperature": 0.2, "max_tokens": max_tokens, "stream": False,
    }
    async with httpx.AsyncClient(timeout=300, trust_env=False) as client:
        r = await client.post(f"{SERVER_URL}/chat/completions", json=payload)
        r.raise_for_status()
        return (r.json()["choices"][0]["message"].get("content") or "").strip()


def build_system_prompt(thinking=None, extra: str = "") -> str:
    entries = load_runbook()
    book = render_runbook(entries)
    now = datetime.now(timezone.utc)
    parts = [BASE_SYSTEM]
    parts.append(f"زمان الان — UTC: {now:%Y-%m-%d %H:%M} | تهران: {now.astimezone(TEHRAN):%Y-%m-%d %H:%M}")
    if book:
        parts.append(
            "تجربیات ثبت‌شده‌ی خودت از کارهای قبلی (به این‌ها اعتماد کن و از اول "
            "از مسیرهایی که قبلاً جواب دادن شروع کن؛ اگه مسیری که اینجا «جواب داد» "
            "ثبت شده این بار fail شد، باز هم ادامه بده و مسیر دیگه‌ای رو امتحان کن):\n" + book)
    if extra:
        parts.append(extra)
    parts.append("/think" if (THINKING if thinking is None else thinking) else "/no_think")
    return "\n\n".join(parts)


# --------------------------------------------------------------------------
# Runbook recording — after a successful multi-step task
# --------------------------------------------------------------------------


async def summarize_and_record(ctx: TaskCtx, user_text: str, injected_tasks: list[str]) -> None:
    """Journal -> one /no_think model call -> structured runbook entry.
    Contradicted old entries gain a strike; two strikes and they are dropped."""
    try:
        lines = []
        for j in ctx.journal:
            mark = "OK " if j["ok"] else "FAIL"
            lines.append(f"[{mark}] {j['tool']}({scrub(j['args'])[:200]}) -> {scrub(j['out'])[:250]}")
        journal_text = "\n".join(lines)
        prompt = (
            "تو خلاصه‌ساز تجربه‌ی یه ایجنت هستی. از روی ژورنال اجرای یه تسک، فقط یه آبجکت JSON برگردون، بدون هیچ متن اضافه:\n"
            '{"task": "شرح کوتاه تسک", "worked": ["مسیر/ابزار/سایتی که جواب داد"], '
            '"failed": ["مسیرهایی که fail شدن"], '
            '"contradicted": ["از لیست تسک‌های قبلی، اون‌هایی که مسیر موفقشون این بار fail شد"]}\n\n'
            f"تسک کاربر: {user_text[:300]}\n\nژورنال:\n{journal_text}\n\n"
            f"تسک‌های قبلی ثبت‌شده: {json.dumps(injected_tasks, ensure_ascii=False)}"
        )
        raw = await plain_completion([
            {"role": "system", "content": "/no_think فقط JSON برگردون."},
            {"role": "user", "content": prompt},
        ])
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        if not m:
            return
        obj = json.loads(m.group(0))
        task = str(obj.get("task", ""))[:200]
        if not task:
            return
        entries = load_runbook()
        # strikes for contradicted entries
        contra = {_norm_task(str(t)) for t in obj.get("contradicted", [])}
        kept = []
        for e in entries:
            if _norm_task(e.get("task", "")) in contra:
                e["strikes"] = int(e.get("strikes", 0)) + 1
                if e["strikes"] >= 2:
                    continue  # expired experience — drop it
            kept.append(e)
        entries = kept
        # de-duplicate: same task replaces the old entry
        nk = _norm_task(task)
        entries = [e for e in entries if _norm_task(e.get("task", "")) != nk]
        entries.append({
            "ts": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "task": task,
            "worked": [str(x)[:250] for x in obj.get("worked", [])][:6],
            "failed": [str(x)[:250] for x in obj.get("failed", [])][:6],
            "strikes": 0,
        })
        save_runbook(entries[-200:])  # hard cap on stored experience
    except Exception:
        pass  # memory recording must never break the bot


# --------------------------------------------------------------------------
# The agent loop (UI-agnostic — the Telegram layer below implements the UI)
# --------------------------------------------------------------------------

_TOOL_LABELS = {
    "run_command": "🖥 اجرای دستور",
    "fetch_url": "🌐 گرفتن صفحه",
    "http_request": "🌐 درخواست HTTP",
    "browser_open": "🧭 بازکردن مرورگر",
    "read_file": "📖 خواندن فایل",
    "write_file": "✍️ نوشتن فایل",
    "list_files": "📂 لیست فایل‌ها",
    "send_file": "📤 ارسال فایل",
    "get_datetime": "🕐 ساعت/تاریخ",
    "remember": "💾 ثبت در حافظه",
    "recall": "🧠 بازیابی حافظه",
}


ACTION_NUDGE = (
    "Stop. You replied WITHOUT calling any tool, but the user's request "
    "requires an action only a tool can perform. Do not explain, "
    "apologize, or offer alternatives in prose. Call the appropriate "
    "tool RIGHT NOW: send_file (with the full path) to send a file; "
    "run_command with curl to upload a file to a hosting site or to run "
    "anything; write_file to create a file; fetch_url/http_request for "
    "web data. If a path might be wrong, locate the file first with "
    "run_command (ls / find). Never claim a limitation you have not "
    "verified through a tool in this task.")


def _needs_action(text: str) -> bool:
    t = (text or "").lower()
    keys = ("بفرست", "ارسال کن", "آپلود", "اجرا کن", "دانلود کن",
            "upload", "send_file", "send the file")
    return any(k in t for k in keys)


async def run_agent(ctx: TaskCtx, history: list[dict], user_text: str, params: dict | None = None) -> str | None:
    """Runs one full task. Returns the final answer text (or None if the task
    produced nothing usable). History is managed by the caller."""
    entries = load_runbook()
    injected_tasks = [e.get("task", "") for e in entries]
    messages = [{"role": "system", "content": build_system_prompt(thinking=(params or {}).get("thinking"), extra=(params or {}).get("system_extra", ""))}] + history + [
        {"role": "user", "content": user_text}]
    status_lines: list[str] = []

    async def set_status(line: str) -> None:
        status_lines.append(line)
        shown = "\n".join(status_lines[-6:])
        await ctx.ui.progress("", "", status=shown)

    final_answer = None
    for step in range(1, MAX_STEPS + 1):
        if ctx.cancelled():
            return None

        async def on_progress(content: str, reasoning: str) -> None:
            await ctx.ui.progress(content, reasoning, status="\n".join(status_lines[-6:]))

        try:
            content, reasoning, tool_calls, _finish = await stream_completion(
                messages, TOOLS, on_progress, params=params)
        except Exception:
            return final_answer  # server hiccup — keep whatever we already have

        if not tool_calls:
            if step == 1 and not ctx.journal and _needs_action(user_text):
                messages.append({"role": "assistant", "content": content or ""})
                messages.append({"role": "user", "content": ACTION_NUDGE})
                continue
            final_answer = visible_text(content) or final_answer
            break

        # Record the assistant turn (with its tool calls) in the conversation.
        messages.append({
            "role": "assistant",
            "content": content or "",
            "tool_calls": [
                {"id": tc["id"], "type": "function",
                 "function": {"name": tc["name"],
                              "arguments": json.dumps(tc["args"], ensure_ascii=False)}}
                for tc in tool_calls],
        })
        for tc in tool_calls:
            if ctx.cancelled():
                return None
            label = _TOOL_LABELS.get(tc["name"], f"🔧 {tc['name']}")
            brief = json.dumps(tc["args"], ensure_ascii=False)
            await set_status(f"قدم {step}: {label} — {brief[:90]}")
            fn = TOOL_FUNCS.get(tc["name"])
            if fn is None:
                result = f"ERROR: unknown tool {tc['name']}"
            else:
                try:
                    result = await fn(ctx, **tc["args"])
                except TypeError as e:
                    result = f"ERROR: bad arguments for {tc['name']}: {e}"
                except Exception as e:
                    result = f"ERROR: {e}"
            ctx.journal.append({
                "tool": tc["name"], "args": brief,
                "ok": not result.startswith("ERROR"),
                "out": result[:400],
            })
            messages.append({"role": "tool", "tool_call_id": tc["id"],
                             "name": tc["name"], "content": result})
        # visible progress lines stay as status; the loop continues
    else:
        if final_answer is None and ctx.journal:
            final_answer = "به سقف قدم‌های این تسک رسیدم و کار کامل نشد. می‌تونی تسک رو کوچیک‌تر یا دقیق‌تر بگی تا دوباره انجامش بدم."

    # Learn from the run — only real multi-step work is worth an entry.
    tool_count = len(ctx.journal)
    if final_answer and tool_count >= 2:
        asyncio.create_task(summarize_and_record(ctx, user_text, injected_tasks))
    return final_answer


# --------------------------------------------------------------------------
# Web layer — custom llama.cpp-style chat layout (full-screen, no box;
# layout as the well-known HF Spaces: settings sidebar + clean chat) over
# the SAME agent core. The agent runs in its own thread with its own event
# loop; the UI talks to it through the shared STATE dict (snapshots + a
# confirm decision slot), so there is no event-loop coupling anywhere.
# The confirm bar is NOT part of the chat template: a slim row under the
# chat that a 1-second timer reveals only while a shell command waits for
# approval. Single-session by design (personal/class tool behind auth).
# --------------------------------------------------------------------------
import threading

import httpx as _httpx

try:
    import gradio as gr
except ImportError:  # pragma: no cover
    raise SystemExit("gradio is not installed — run the setup step first "
                     "(pip install gradio).")

FA_CSS = """
/* Chat content: each message follows its own direction (unicode-bidi:
   plaintext) so Persian lines lay out correctly and mixed content with
   numbers/URLs/English does not scramble; code stays LTR and isolated. */
#fa-chat { direction: rtl; }
#fa-chat .message, #fa-chat .bubble, #fa-chat p, #fa-chat li {
    unicode-bidi: plaintext; text-align: right;
}
#fa-chat pre, #fa-chat code, #fa-chat table {
    direction: ltr; text-align: left; unicode-bidi: isolate;
}
#fa-input textarea, #fa-input2 textarea { direction: rtl; text-align: right; }
/* Use the full width of the window (desktop included). Gradio caps the
   app in a centered column through several nested wrappers — release
   them all and keep only a hair of side padding. */
.gradio-container { max-width: 100% !important; padding-left: 6px !important;
                   padding-right: 6px !important; direction: rtl; }
/* Gradio's own footer strip is English/LTR — hide it. */
.gradio-container footer { display: none !important; }
.gradio-container .app, .gradio-container .main,
.gradio-container .wrap { max-width: 100% !important; }
/* The conversation fills the screen; no box drawn around it. */
#fa-chat { border: none !important; background: transparent !important;
           box-shadow: none !important;
           height: max(620px, calc(100vh - 380px)) !important; }
/* The screenshot preview lives inside an accordion that starts closed, so
   Gradio mis-measures it and squeezes the photo into a thin strip. Force
   the img back to full width with its natural aspect ratio. */
#shot-img img { width: 100% !important; height: auto !important;
                max-height: 280px; object-fit: contain; }
"""

WEB_USER = os.environ.get("WEB_USER", "")
WEB_PASS = os.environ.get("WEB_PASS", "")
WEB_PORT = int(os.environ.get("WEB_PORT", "7860"))

STATE: dict = {
    "running": False, "finished": True, "ctx": None,
    "content": "", "reasoning": "", "status": "",
    "photo": None, "file": None,
    "confirm_command": None, "decision": None,
    "final": None, "error": None,
}


class WebUI:
    """Same UI contract the tools already use (progress / confirm / photo /
    file) — implemented against the shared STATE instead of Telegram."""

    def __init__(self, state: dict):
        self.state = state

    async def progress(self, content: str, reasoning: str, status: str = "") -> None:
        st = self.state
        if content:
            st["content"] = content
        if reasoning:
            st["reasoning"] = reasoning
        if status:
            st["status"] = status

    async def confirm(self, command: str) -> str:
        st = self.state
        st["decision"] = None
        st["confirm_command"] = command[:1500]
        waited = 0.0
        try:
            while st["decision"] is None:
                await asyncio.sleep(0.15)
                waited += 0.15
                if waited >= CONFIRM_TIMEOUT:
                    return "deny"
            return st["decision"]
        finally:
            st["confirm_command"] = None
            st["decision"] = None

    async def photo(self, path, caption: str = "") -> None:
        self.state["photo"] = str(path)

    async def file(self, path) -> None:
        self.state["file"] = str(path)


def _run_agent_thread(user_text: str, params: dict) -> None:
    st = STATE

    async def _main() -> None:
        chat_id = 0
        history = histories.setdefault(chat_id, [])
        ctx = TaskCtx(WebUI(st), chat_id)
        ctx.confirm_mode = bool(params.get("confirm_mode", True))
        st["ctx"] = ctx
        try:
            answer = await run_agent(ctx, list(history), user_text, params=params)
            # run_agent fires the runbook recording as a background task;
            # asyncio.run() would cancel it on exit — drain it first so the
            # experience actually lands in the runbook.
            pending = [t for t in asyncio.all_tasks()
                       if t is not asyncio.current_task()]
            if pending:
                try:
                    await asyncio.wait_for(
                        asyncio.gather(*pending, return_exceptions=True),
                        timeout=180)
                except asyncio.TimeoutError:
                    pass
        except Exception:
            answer = None
        if ctx.cancelled():
            st["error"] = "⏹ تسک لغو شد."
            return
        if not answer:
            st["error"] = ("الان به نتیجه نرسیدم — یا سرور مدل جواب نداد یا تسک "
                           "وسط کار قطع شد. دوباره بفرست.")
            return
        history.append({"role": "user", "content": user_text})
        history.append({"role": "assistant", "content": answer})
        while len(history) > HISTORY_TURNS * 2 or (
            sum(len(m["content"]) for m in history) > MAX_HISTORY_CHARS
            and len(history) > 2
        ):
            del history[:2]
        _save_histories()
        st["final"] = answer

    try:
        asyncio.run(_main())
    finally:
        st["running"] = False
        st["finished"] = True
        st["ctx"] = None


def _assistant_text(st: dict) -> str:
    if st["final"]:
        return st["final"]
    if st["error"]:
        return st["error"]
    if st["confirm_command"]:
        return (f"⏸ منتظر تایید توئم — مدل می‌خواد این دستور رو اجرا کنه:\n\n"
                f"```\n{st['confirm_command']}\n```\n\n"
                "دکمه‌های تایید زیر چت هستن.")
    vis = visible_text(st["content"]) if st["content"] else ""
    if vis:
        return vis
    if st["status"]:
        return "⏳ در حال انجام:\n" + st["status"]
    if st["reasoning"]:
        return THINKING_TEXT + "\n\n" + st["reasoning"][-1200:]
    return WORKING_TEXT


def chat_fn(message, history):
    """gr.ChatInterface handler — a sync generator bridging to the agent
    thread through STATE. Generation settings come from the SETTINGS dict
    (kept current by the Settings panel at the bottom of the page), so the
    panel's position in the layout is free."""
    st = STATE
    message = (message or "").strip()
    if not message:
        yield ""
        return
    if st["running"]:
        yield "⏳ دارم روی تسک قبلی کار می‌کنم — صبر کن تموم بشه یا توقف رو بزن."
        return
    st.update({"content": "", "reasoning": "", "status": "", "photo": None,
               "file": None, "final": None, "error": None, "decision": None,
               "confirm_command": None, "running": True, "finished": False})
    s = SETTINGS
    params = {"temperature": float(s["temperature"]),
              "top_p": float(s["top_p"]), "top_k": int(s["top_k"]),
              "max_tokens": int(s["max_tokens"]),
              "repeat_penalty": float(s["repeat_penalty"]),
              "thinking": bool(s["thinking"]),
              "system_extra": s["system_extra"] or "",
              "confirm_mode": bool(s["confirm_mode"])}
    threading.Thread(target=_run_agent_thread, args=(message, params),
                     daemon=True).start()
    yield WORKING_TEXT
    while not st["finished"]:
        yield _assistant_text(st)
        time.sleep(0.3)
    final = _assistant_text(st)
    if st["photo"] and st["final"]:
        # Show the browser screenshot in the conversation itself, as a
        # LINKED image: tapping it opens the full-size shot in a new tab
        # (a bare markdown image in a chatbot bubble is not tappable).
        url = f"/gradio_api/file={st['photo']}"
        final += (f"\n\n[![📸 اسکرین‌شات مرورگر]({url})]({url})\n\n"
                  "📸 برای دیدن اسکرین‌شات در اندازه‌ی کامل، روش بزن.")
    if st["file"] and st["final"]:
        # Same for sent files: the download lands in the conversation,
        # not only in the tools panel where nobody looks for it.
        furl = f"/gradio_api/file={st['file']}"
        final += f"\n\n📤 فایل ارسالی: [{Path(st['file']).name}]({furl}) — برای دانلود روش بزن."
    yield final


def _poll():
    """Timer tick: reveal the confirm bar while a command waits; refresh
    the screenshot / file slots in the sidebar."""
    st = STATE
    if st["confirm_command"]:
        row = gr.update(visible=True)
        md = (f"⚠️ مدل می‌خواد این دستور رو اجرا کنه:\n\n"
              f"```\n{st['confirm_command']}\n```\n\nاجازه می‌دی؟")
    else:
        row = gr.update(visible=False)
        md = ""
    fupd = gr.update(visible=bool(st["file"]), value=st["file"])
    return row, md, fupd


def on_decide(choice: str):
    STATE["decision"] = choice
    return gr.update(visible=False)


def on_stop():
    ctx = STATE.get("ctx")
    if ctx is not None:
        ctx.cancel.set()
        if ctx.proc is not None:
            try:
                ctx.proc.kill()
            except Exception:
                pass


def on_memory() -> str:
    entries = load_runbook()
    if not entries:
        return "هنوز تجربه‌ای ثبت نکردم. یه کار چندمرحله‌ای بده تا یاد بگیرم."
    return "🧠 تجربه‌های ثبت‌شده:\n\n" + render_runbook(entries, cap=6000)


def on_wipe() -> str:
    save_runbook([])
    histories.pop(0, None)
    _save_histories()
    return "حافظه‌ی دائمی (ران‌بوک) و گفتگو پاک شد. از صفر شروع می‌کنم."


def on_reset() -> str:
    histories.pop(0, None)
    _save_histories()
    return "گفتگو از سمت سرور از اول شروع شد؛ برای خالی شدن صفحه، چت رو پاک کن یا صفحه رو رفرش کن."


def on_status() -> str:
    health = SERVER_URL[:-3] + "/health" if SERVER_URL.endswith("/v1") else SERVER_URL + "/health"
    try:
        with _httpx.Client(timeout=5, trust_env=False) as c:
            ok = c.get(health).status_code == 200
    except Exception:
        ok = False
    entries = load_runbook()
    return (f"سرور مدل: {'✅ روشن' if ok else '❌ خاموش/در حال بالا اومدن'}\n"
            f"مدل: {MODEL} | کانتکست سرور: 32K\n"
            f"تجربه‌های ثبت‌شده: {len(entries)}\n"
            f"حافظه: {'دائمی (Drive)' if 'drive' in str(DATA_HOME).lower() else 'لوکال این سشن'}")


def _watchdog() -> None:
    """Same contract as the bot's: if llama-server dies, relaunch it through
    server_cmd.sh (written by the start step). Runs in a daemon thread."""
    health = SERVER_URL[:-3] + "/health" if SERVER_URL.endswith("/v1") else SERVER_URL + "/health"
    cmd_file = BASE_DIR / "server_cmd.sh"
    fails = 0
    last_launch = 0.0
    while True:
        time.sleep(30)
        try:
            with _httpx.Client(timeout=5, trust_env=False) as c:
                ok = c.get(health).status_code == 200
        except Exception:
            ok = False
        if ok:
            fails = 0
            continue
        fails += 1
        if fails >= 2 and cmd_file.exists() and time.time() - last_launch > 180:
            try:
                subprocess.Popen(["bash", str(cmd_file)],
                                 stdout=subprocess.DEVNULL,
                                 stderr=subprocess.DEVNULL,
                                 start_new_session=True)
                last_launch = time.time()
                fails = 0
            except Exception:
                pass


DEFAULT_EXTRA = (
    "You are an agent assistant running on a Linux server with tool access. "
    "For multi-step tasks, use your tools and report results based on real "
    "tool output, never on guesses. Always reply in the same language the "
    "user wrote in. You are inside a web page, not Telegram — you have no "
    "access to Telegram or any other messenger, and never mention Telegram. "
    "When the user asks for a file, create it and send it immediately with "
    "send_file; the file reaches the user as a download in this very "
    "conversation. When the user asks you to send a file, call send_file "
    "with the full file path right away — no explanations, excuses, or "
    "how-to advice instead of sending. Before saying something cannot be "
    "done or needs an API key, actually try keyless routes and alternative "
    "sources with your tools first. You can also upload files to external "
    "file-hosting sites when the user asks: use run_command with curl "
    "against the site's upload API (for example tmpfiles.org's "
    "/api/v1/upload endpoint, 0x0.st, or catbox.moe) and report the real "
    "download link from the tool output. For casual small talk such as greetings, "
    "reply briefly, naturally, and with variety — never repeat your "
    "previous reply verbatim.")

# Live generation settings, mirrored from the Settings panel (which sits at
# the BOTTOM of the page at his request). chat_fn reads from here, so the
# panel's position is decoupled from the ChatInterface wiring.
SETTINGS = {"system_extra": DEFAULT_EXTRA, "max_tokens": 1024,
            "temperature": TEMPERATURE, "top_p": TOP_P, "top_k": 40,
            "repeat_penalty": 1.0, "thinking": THINKING, "confirm_mode": True}


def _bind_setting(comp, key, cast=None):
    def _set(v):
        SETTINGS[key] = cast(v) if cast else v
    comp.change(_set, inputs=[comp])


_PLACEHOLDER_JS = """
() => {
  const fix = () => {
    const ta = document.querySelector('#fa-input textarea');
    if (ta && ta.getAttribute('placeholder') !== 'پیامت رو بنویس...') {
      ta.setAttribute('placeholder', 'پیامت رو بنویس...');
    }
  };
  fix();
  setInterval(fix, 1000);
}
"""


def build_demo():
    # Chat-first: the chat has a FIXED height and never resizes; the page
    # itself scrolls, so opening the panels below pushes the page longer
    # instead of squeezing the chat (his explicit request — with
    # fill_height, expanding an accordion stole height from the chat).
    # Tools/memory right under it; Settings at the very bottom.
    with gr.Blocks(title="ایجنت نمترون") as demo:
        chat = gr.ChatInterface(
            fn=chat_fn,
            title="Nemotron Nano 12B",
            chatbot=gr.Chatbot(label="", elem_id="fa-chat", show_label=False,
                               height=620),
            textbox=gr.Textbox(elem_id="fa-input", show_label=False,
                               placeholder="پیامت رو بنویس..."),
        )
        with gr.Row(visible=False) as confirm_row:
            confirm_md = gr.Markdown("")
            btn_once = gr.Button("✅ اجرا کن", variant="primary")
            btn_deny = gr.Button("❌ نکن", variant="stop")
        # The sent-file slot lives OUTSIDE any accordion, directly under
        # the chat, hidden until a file is actually sent. Inside the
        # (initially closed) tools accordion its download tap was dead on
        # his phone — same accordion-init disease that broke the image.
        file_out = gr.File(label="📤 فایل ارسالی ایجنت — برای دانلود روش بزن",
                           visible=False)
        with gr.Accordion("🛠 ابزارها و حافظه", open=False):
            with gr.Row():
                btn_stop = gr.Button("⏹ توقف تسک")
                btn_memory = gr.Button("📚 حافظه")
                btn_status = gr.Button("📊 وضعیت")
                btn_reset = gr.Button("🔄 ریست")
                btn_wipe = gr.Button("🧹 پاک‌کردن حافظه")
            panel = gr.Textbox(show_label=False, lines=6, interactive=False)
            # No screenshot preview slot here: inside this initially
            # closed accordion it never rendered on his phone. Screenshots
            # are delivered inside the chat itself (linked image — proven).
        with gr.Accordion("⚙️ تنظیمات — پارامترهای تولید", open=False):
            system_prompt = gr.Textbox(label="پرامپت سیستم",
                                       value=DEFAULT_EXTRA, lines=4)
            with gr.Row():
                max_tokens_sl = gr.Slider(256, 8192, value=1024, step=64,
                                          label="حداکثر توکن خروجی")
                temp_sl = gr.Slider(0, 1.5, value=TEMPERATURE, step=0.05,
                                    label="دما (Temperature)")
                topp_sl = gr.Slider(0.05, 1.0, value=TOP_P, step=0.01, label="Top-p")
            with gr.Row():
                topk_sl = gr.Slider(0, 100, value=40, step=1, label="Top-k")
                reppen_sl = gr.Slider(0.5, 2.0, value=1.0, step=0.05,
                                      label="جریمه‌ی تکرار")
                thinking_chk = gr.Checkbox(value=THINKING, label="🧠 تفکر (Thinking)")
                confirm_chk = gr.Checkbox(
                    value=True,
                    label="🔒 تایید قبل از اجرای هر دستور (ترمینال) — تیک داشته باشه، هر دستور اول ازت تایید می‌گیره؛ برداری، دستورها مستقیم اجرا می‌شن")
            _bind_setting(system_prompt, "system_extra")
            _bind_setting(max_tokens_sl, "max_tokens", int)
            _bind_setting(temp_sl, "temperature", float)
            _bind_setting(topp_sl, "top_p", float)
            _bind_setting(topk_sl, "top_k", int)
            _bind_setting(reppen_sl, "repeat_penalty", float)
            _bind_setting(thinking_chk, "thinking", bool)
            _bind_setting(confirm_chk, "confirm_mode", bool)
        timer = gr.Timer(1.0)
        timer.tick(_poll, outputs=[confirm_row, confirm_md, file_out])
        btn_once.click(lambda: on_decide("once"), outputs=[confirm_row])
        btn_deny.click(lambda: on_decide("deny"), outputs=[confirm_row])
        btn_stop.click(on_stop)
        btn_memory.click(on_memory, outputs=[panel])
        btn_status.click(on_status, outputs=[panel])
        btn_wipe.click(on_wipe, outputs=[panel])
        btn_reset.click(on_reset, outputs=[panel])
        demo.load(on_status, outputs=[panel])
        demo.load(None, None, None, js=_PLACEHOLDER_JS)
    return demo


def main() -> None:
    threading.Thread(target=_watchdog, daemon=True).start()
    if WEB_USER and WEB_PASS:
        auth = (WEB_USER, WEB_PASS)
        print(f"🔐 login is ON — user: {WEB_USER}")
    else:
        auth = None
        print("🔓 login is OFF (نام کاربری/پسورد ست نشده) — the page opens "
              "without a login, for anyone with the link.")
    try:
        demo = build_demo()
        demo.queue()
        demo.launch(server_name="0.0.0.0", server_port=WEB_PORT,
                    share=True, auth=auth, css=FA_CSS,
                    allowed_paths=["/content", str(WORK_DIR), str(DATA_HOME)])
    except Exception as e:
        # The public tunnel is a convenience, not a requirement. A failed
        # share check can raise AFTER the local server already started —
        # if the port answers, just stay alive and let the Colab proxy URL
        # (printed by the notebook) reach the page.
        print(f"⚠️ ساخت لینک عمومی نشد ({type(e).__name__}) — صفحه رو با لینک "
              "پروکسی کولب باز کن.")
        import urllib.error
        import urllib.request
        alive = False
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{WEB_PORT}/", timeout=5)
            alive = True
        except urllib.error.HTTPError:
            alive = True  # a 401/403 still proves the server is up
        except Exception:
            alive = False
        if alive:
            print("✅ سرور لوکال بالاست و داره سرویس می‌ده.")
            while True:
                time.sleep(3600)
        demo = build_demo()
        demo.queue()
        demo.launch(server_name="0.0.0.0", server_port=WEB_PORT,
                    share=False, auth=auth, css=FA_CSS,
                    allowed_paths=["/content", str(WORK_DIR), str(DATA_HOME)])


if __name__ == "__main__":
    main()

Writing web_app.py


<div dir="rtl">

## ▶️ استارت سرور مدل
این سلول ستاپ + سرور رو بالا میاره و گرمش می‌کنه. وقتی نوشت `MODEL SERVER IS RUNNING` یعنی آماده‌ست.
</div>

<div dir="rtl">

## 📖  اجرای استارت

این سلول ستاپ + لانچ + وارم‌آپ را یکجا انجام می‌دهد. خروجی سالم دو خط دارد: `Setup OK` و در پایان `MODEL SERVER IS RUNNING`.

- زمان مورد انتظار روی رانتایم تازه برای این مدل: **حدود ۲ تا ۳ دقیقه (دانلود ۷.۵ گیگی + لود سبک)**.
- اگر سرور موقع لود بمیرد، خود سلول دمِ لاگ را چاپ می‌کند: همیشه اول لاگ، بعد حدس.

</div>


In [ ]:
!bash start_server.sh

==> parallel setup: binary + model (7.5 GB) + python deps + tor
==> browser install running in BACKGROUND — not blocking the server (log: .setup-logs/browser.log)
==> CUDA library fix-ups
  linked libcuda.so.1 -> /usr/lib64-nvidia/libcuda.so.1
  linked libnvidia-ptxjitcompiler.so.1 -> /usr/lib64-nvidia/libnvidia-ptxjitcompiler.so.1
  linked libcublas.so.12 -> /usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib/libcublas.so.12
  linked libcublasLt.so.12 -> /usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib/libcublasLt.so.12
  CUDA library check OK (ldd clean)
    [binary] built with GNU 13.3.0 for Linux x86_64
    [model] -rw-r--r-- 1 root root 7.0G Oct 28 01:47 models/nvidia_NVIDIA-Nemotron-Nano-12B-v2-Q4_K_M.gguf
    [pip] python deps OK
    [browser] Chrome for Testing 156.0.8078.4 (playwright chromium v1248) downloaded to /root/.cache/ms-playwright/chromium-1248
✅ Setup OK in 90s
==> starting llama-server (Nemotron Nano 12B v2, Q4_K_M, T4)...
==> waiting for the model t

<div dir="rtl">

## 🌐 لینک صفحه‌ی ایجنت (پروکسی کولب)
اگه لینک عمومی‌ای که سلول بعد می‌ده باز نشد، این سلول لینک پروکسی خود کولب رو می‌ده (فقط با همین اکانت گوگلت باز می‌شه).

</div>

<div dir="rtl">

## 📖  لینک صفحه‌ی ایجنت

این سلول لینک **پروکسی خود کولب** را می‌سازد (`proxyPort`). نکته‌اش: این لینک فقط در مرورگری باز می‌شود که با **همان اکانت گوگلی** لاگین باشد که رانتایم مال اوست.

راه دوم، لینک عمومی `gradio.live` است که سلول اجرای وب چاپ می‌کند و برای نمایش راحت‌تر است؛ اگر در سلول اول پسورد ست کرده باشید، همان پسورد دربان این صفحه است.

</div>


In [ ]:
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(7860)"))

https://7860-gpu-t4-s-usw4a2-34m0h5i7fbagu-a.us-west4.prod.r.colab.dev


<div dir="rtl">

## 🤖 اجرای ایجنت وب
این سلول تا وقتی ایجنت روشنه در حال اجرا می‌مونه . لینک صفحه (عمومی + لوکال) تو خروجی چاپ می‌شه.

</div>

<div dir="rtl">

## 📖 توضیح آموزشی — اجرای ایجنت وب (سلول بعدی)

این سلول تا وقتی صفحه روشن است «در حال اجرا» می‌ماند — طبیعی است، سرور وب داخل همین سلول زندگی می‌کند. لینک لوکال و لینک عمومی در خروجی‌اش چاپ می‌شود.

- از این لحظه واچ‌داگ فعال است و وضعیت سرور مدل از دکمه‌ی **📊 وضعیت** در پنل «ابزارها و حافظه»‌ی خود صفحه دیده می‌شود.

</div>


In [ ]:
!python3 web_app.py

🔓 login is OFF (نام کاربری/پسورد ست نشده) — the page opens without a login, for anyone with the link.
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://fe5df5ccba3baa7f20.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Keyboard interruption in main thread... closing server.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 3957, in block_thread
    time.sleep(0.1)
    ~~~~~~~~~~^^^^^
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/nemotron-web/web_app.py", line 1264, in <module>
    main()
    ~~~~^^
  File "/content/nemotron-web/web_app.py", line 1232, in main
    demo.launch(server_name="0.0.0.0", ser